In [48]:

import pandas as pd
import argparse
import os
import sys
import mysql.connector
from dotenv import load_dotenv

load_dotenv()


True

In [49]:

DATA_DIR = "data"
SUB_IN = 1
SUB_OUT = 2
STARTING_LINEUP_PLAY_ID = 1
PLAYERS_PER_TEAM = 5
TECHNICAL = "Technical"


In [50]:

ID_COLS_PBP  = ["event_id", "play_id", "play_sequence", "period",
                "home_team_id", "away_team_id", "play_team_id"]
ID_COLS_PBPP = ["event_id", "play_id", "play_sequence", "period", "player_id", "team_id"]
ID_COLS_ROS  = ["event_id", "team_id", "player_id"]

def load(data_dir):
    pbp  = pd.read_excel(f"{data_dir}/pbp.xlsx")
    pbpp = pd.read_excel(f"{data_dir}/pbp-players.xlsx")
    ros  = pd.read_excel(f"{data_dir}/rosters.xlsx")

    pbpp = pbpp[pbpp.player_id.notna()].copy()

    pbp[ID_COLS_PBP]   = pbp[ID_COLS_PBP].astype(int)
    pbpp[ID_COLS_PBPP] = pbpp[ID_COLS_PBPP].astype(int)
    ros[ID_COLS_ROS]   = ros[ID_COLS_ROS].astype(int)

    pbp  = pbp.sort_values(["event_id", "play_id", "play_sequence"]).reset_index(drop=True)
    pbpp = pbpp.sort_values(["event_id", "play_id", "play_sequence"]).reset_index(drop=True)
    ros  = ros.sort_values(["event_id", "team_id", "player_id"]).reset_index(drop=True)
    return pbp, pbpp, ros



In [51]:

pbp, pbpp, ros = load(DATA_DIR)
pbp.shape, pbpp.shape, ros.shape


((1011, 30), (1284, 38), (66, 12))

In [52]:
def attach_roster_team(pbpp: pd.DataFrame, ros: pd.DataFrame) -> pd.DataFrame:
    roster_team = ros.set_index(["event_id", "player_id"])["team_id"].rename("roster_team_id")
    out = pbpp.join(roster_team, on=["event_id", "player_id"])
    out["team_id"] = out.pop("roster_team_id").astype(int)
    return out

In [53]:
pbpp = attach_roster_team(pbpp, ros)
print(pbpp.team_id.isna().sum(), "missing team_ids")
pbpp.loc[pbpp.play_id == 149, ["player_id", "last_name", "team_abbr", "team_id"]]

0 missing team_ids


,player_id,last_name,team_abbr,team_id
194,468895,Green,GS,9
195,173004,Iguodala,GS,9
835,845564,Booker,Hou,21


In [ ]:
OPENER_COLS = ["event_id", "period", "team_id", "player_id"]

def period_openers(pbpp: pd.DataFrame) -> pd.DataFrame:
    """Q1 from the Starting Lineup rows; Q2+ inferred from each player's first appearance."""

    starter = pbpp.loc[pbpp.play_id == STARTING_LINEUP_PLAY_ID, OPENER_COLS]

    evidence = pbpp[(pbpp.period >= 2) & (pbpp.play_detail != TECHNICAL)]
    first_row = evidence.drop_duplicates(subset=OPENER_COLS, keep="first")
    checked_in = (first_row.play_event == "Substitution") & (first_row.sequence == SUB_IN)
    inferred = first_row.loc[~checked_in, OPENER_COLS]

    openers = pd.concat([starter, inferred], ignore_index=True)
    return openers

In [77]:

openers = period_openers(pbpp)
openers.sort_values(["event_id", "period", "team_id", "player_id"])

,event_id,period,team_id,player_id
0,1947160,1,2,280587
9,1947160,1,2,469089
8,1947160,1,2,551768
7,1947160,1,2,883436
6,1947160,1,2,937647
...,...,...,...,...
70,1947312,4,21,467468
78,1947312,4,21,513526
71,1947312,4,21,567497
76,1947312,4,21,920066


In [78]:
counts = openers.groupby(["event_id", "period", "team_id"]).size().reset_index(name="players")
print("groups:", len(counts), "| all 5:", (counts.players == 5).all(), "| rows:", len(openers))
counts

groups: 16 | all 5: True | rows: 80


,event_id,period,team_id,players
0,1947160,1,2,5
1,1947160,1,9,5
2,1947160,2,2,5
3,1947160,2,9,5
4,1947160,3,2,5
5,1947160,3,9,5
6,1947160,4,2,5
7,1947160,4,9,5
8,1947312,1,10,5
9,1947312,1,21,5


In [ ]:
def openers_wide(openers: pd.DataFrame, ros: pd.DataFrame) -> pd.DataFrame:
    """One row per game per team"""

    named = openers.merge(ros[["event_id", "player_id", "name"]], on=["event_id", "player_id"])
    named["label"] = named.name + " (" + named.player_id.astype(str) + ")"

    per_period = (
        named.sort_values("name")
        .groupby(["event_id", "team_id", "period"])["label"]
        .agg(", ".join)
        .unstack("period")
        .rename(columns=lambda p: f"Q{p}")
        .reset_index()
    )
    return per_period

In [79]:
openers_stacked = openers_wide(openers,ros)
openers_stacked

period,event_id,team_id,Q1,Q2,Q3,Q4
0,1947160,2,"Al Horford (280587), Jaylen Brown (883436), Ja...","Aron Baynes (296379), Daniel Theis (1066725), ...","Al Horford (280587), Jaylen Brown (883436), Ja...","Aron Baynes (296379), Jayson Tatum (937647), M..."
1,1947160,9,"Draymond Green (468895), Kevin Durant (329525)...","Andre Iguodala (173004), David West (57934), D...","Draymond Green (468895), Kevin Durant (329525)...","Andre Iguodala (173004), David West (57934), D..."
2,1947312,10,"Chris Paul (229598), Clint Capela (830647), Ja...","Chris Paul (229598), Eric Gordon (401085), Nen...","Chris Paul (229598), Clint Capela (830647), Ja...","Chris Paul (229598), Eric Gordon (401085), Jam..."
3,1947312,21,"Devin Booker (845564), Greg Monroe (467468), M...","Alex Len (616982), Devin Booker (845564), Josh...","Devin Booker (845564), Greg Monroe (467468), M...","Dragan Bender (920066), Greg Monroe (467468), ..."


In [95]:
def sub_events(pbpp: pd.DataFrame) -> pd.DataFrame:
    """One row per substitution: event_id, play_id, team_id, player_in, player_out."""

    subs = pbpp[pbpp.play_event == "Substitution"]
    pivoted = (
        subs.pivot_table(
            index=["event_id", "play_id", "period", "team_id"],
            columns="sequence",
            values="player_id",
            aggfunc="first"
        )
        .rename(columns={SUB_IN: "player_in", SUB_OUT: "player_out"})
        .astype(int)
        .reset_index()
    )
    pivoted.columns.name = None
    return pivoted[["event_id", "play_id", "period", "team_id", "player_in", "player_out"]]

In [96]:
subs = sub_events(pbpp)
subs

,event_id,play_id,period,team_id,player_in,player_out
0,1947160,41,1,2,697132,937647
1,1947160,59,1,9,173004,468895
2,1947160,66,1,2,792415,469089
3,1947160,67,1,9,263228,457611
4,1947160,96,1,2,1066725,551768
...,...,...,...,...,...,...
99,1947312,409,4,10,908263,329715
100,1947312,437,4,21,616982,467468
101,1947312,438,4,21,845574,567497
102,1947312,439,4,10,242307,395388


In [97]:
OUT_COLS = ["event_id", "play_id", "player_id", "team_id", "period", "is_home"]


def add_is_home(long: pd.DataFrame, pbp: pd.DataFrame) -> pd.DataFrame:
    """Attach is_home by comparing team_id to the game's home_team_id."""

    home = pbp[["event_id", "home_team_id"]].drop_duplicates()
    out = long.merge(home, on="event_id")
    out["is_home"] = (out.team_id == out.home_team_id).astype(int)
    result = out[OUT_COLS].sort_values(OUT_COLS[:4]).reset_index(drop=True)
    return result

In [98]:


def walk_plays(pbp: pd.DataFrame, openers: pd.DataFrame, subs: pd.DataFrame) -> pd.DataFrame:
    """One row per play per on-court player"""

    plays = pbp[["event_id", "play_id", "period"]].drop_duplicates().sort_values(["event_id", "play_id"])

    rows = []
    for (eid, per, team), five in openers.groupby(["event_id", "period", "team_id"]):
        on_court = set(five.player_id)
        team_subs = subs[(subs.event_id == eid) & (subs.period == per) & (subs.team_id == team)].set_index("play_id")
        period_plays = plays[(plays.event_id == eid) & (plays.period == per)]

        for play in period_plays.itertuples(index=False):
            if play.play_id in team_subs.index:
                on_court.remove(team_subs.at[play.play_id, "player_out"])
                on_court.add(team_subs.at[play.play_id, "player_in"])

            rows.extend((eid, play.play_id, p, team, per) for p in on_court)

    long = pd.DataFrame(rows, columns=OUT_COLS[:-1])
    return add_is_home(long, pbp)



In [99]:
on_court = walk_plays(pbp, openers, subs)
print(on_court.shape, "| 10 per play:", (on_court.groupby(["event_id", "play_id"]).size() == 10).all())
on_court[(on_court.event_id == 1947160) & on_court.play_id.between(40, 42)].merge(ros[["event_id", "player_id", "name"]])

(9930, 6) | 10 per play: True


,event_id,play_id,player_id,team_id,period,is_home,name
0,1947160,40,214168,9,1,0,Zaza Pachulia
1,1947160,40,280587,2,1,1,Al Horford
2,1947160,40,329525,9,1,0,Kevin Durant
3,1947160,40,338365,9,1,0,Stephen Curry
4,1947160,40,457611,9,1,0,Klay Thompson
5,1947160,40,468895,9,1,0,Draymond Green
6,1947160,40,469089,2,1,1,Marcus Morris
7,1947160,40,551768,2,1,1,Kyrie Irving
8,1947160,40,883436,2,1,1,Jaylen Brown
9,1947160,40,937647,2,1,1,Jayson Tatum


In [100]:
def validate(on_court: pd.DataFrame, pbp: pd.DataFrame, ros: pd.DataFrame) -> None:
    """Raise AssertionError on the first broken invariant."""

    for eid, g in pbp.groupby("event_id"):
        missing = set(g.play_id) - set(on_court.loc[on_court.event_id == eid, "play_id"])
        assert not missing, f"game {eid}: {len(missing)} play_ids missing, e.g. {sorted(missing)[:5]}"

    per_play = on_court.groupby(["event_id", "play_id"]).size()
    assert (per_play == 10).all(), f"plays without 10 players:\n{per_play[per_play != 10].head()}"

    per_team = on_court.groupby(["event_id", "play_id", "team_id"]).size()
    assert (per_team == 5).all(), f"team-plays without 5 players:\n{per_team[per_team != 5].head()}"

    on_roster = on_court.merge(ros[["event_id", "player_id"]], how="left", indicator=True)
    assert (on_roster._merge == "both").all(), f"players not on roster:\n{on_roster[on_roster._merge != 'both'].head()}"

    dupes = on_court.duplicated(["event_id", "play_id", "player_id"]).sum()
    assert dupes == 0, f"{dupes} duplicate (event, play, player) rows"

    print(f"validate: 5/5 passed — {len(on_court)} rows, {len(per_play)} plays, {on_court.event_id.nunique()} games")

In [101]:
validate(on_court, pbp, ros)

validate: 5/5 passed — 9930 rows, 993 plays, 2 games


In [119]:
DDL = """
CREATE TABLE IF NOT EXISTS pbp_players_on_court (
  event_id   INT        NOT NULL,
  play_id    INT        NOT NULL,
  player_id  INT        NOT NULL,
  team_id    INT        NOT NULL,
  period     TINYINT    NOT NULL,
  is_home    TINYINT(1) NOT NULL,
  updated_at TIMESTAMP  NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
  PRIMARY KEY (event_id, play_id, player_id),
  KEY ix_player (event_id, player_id),
  KEY ix_team   (event_id, team_id, play_id)
)
"""


In [120]:
DDL_PBP = """
CREATE TABLE IF NOT EXISTS pbp (
  season           SMALLINT     NOT NULL,
  date             DATE         NOT NULL,
  event_id         INT          NOT NULL,
  home_team_id     INT          NOT NULL,
  home_team_abbr   VARCHAR(5)   NOT NULL,
  away_team_id     INT          NOT NULL,
  away_team_abbr   VARCHAR(5)   NOT NULL,
  play_id          INT          NOT NULL,
  play_sequence    TINYINT      NOT NULL,
  period           TINYINT      NOT NULL,
  clock_minutes    TINYINT      NOT NULL,
  clock_seconds    TINYINT      NOT NULL,
  sec_left         SMALLINT     NOT NULL,
  play_team_id     INT          NOT NULL,
  points_scored    TINYINT          NULL,
  play_event_id    TINYINT      NOT NULL,
  play_event       VARCHAR(40)      NULL,
  play_detail_id   SMALLINT         NULL,
  play_detail      VARCHAR(60)      NULL,
  is_blocked       TINYINT(1)   NOT NULL,
  distance         TINYINT      NOT NULL,
  is_fast_break    TINYINT(1)       NULL,   -- NULL on Starting Lineup rows
  is_in_the_paint  TINYINT(1)       NULL,
  is_off_turnover  TINYINT(1)       NULL,
  is_second_chance TINYINT(1)       NULL,
  away_score       SMALLINT     NOT NULL,
  home_score       SMALLINT     NOT NULL,
  away_fouls       TINYINT      NOT NULL,
  home_fouls       TINYINT      NOT NULL,
  play_text        VARCHAR(255) NOT NULL,
  PRIMARY KEY (event_id, play_id, play_sequence),
  KEY ix_pbp_team (event_id, play_team_id, play_id)
)
"""

In [121]:
DDL_PBP_PLAYERS = """
CREATE TABLE IF NOT EXISTS pbp_players (
  season           SMALLINT     NOT NULL,
  date             DATE         NOT NULL,
  event_id         INT          NOT NULL,
  home_team_id     INT          NOT NULL,
  home_team_abbr   VARCHAR(5)   NOT NULL,
  away_team_id     INT          NOT NULL,
  away_team_abbr   VARCHAR(5)   NOT NULL,
  play_id          INT          NOT NULL,
  play_sequence    TINYINT      NOT NULL,
  period           TINYINT      NOT NULL,
  clock_minutes    TINYINT      NOT NULL,
  clock_seconds    TINYINT      NOT NULL,
  player_id        INT          NOT NULL,
  first_name       VARCHAR(40)  NOT NULL,
  last_name        VARCHAR(40)  NOT NULL,
  team_id          INT          NOT NULL,   -- roster team, not the feed's
  team_abbr        VARCHAR(5)   NOT NULL,   -- feed's; wrong on play 149 (Booker) — keep for the writeup
  score            TINYINT          NULL,   -- player's running points; NULL on 346 rows
  fouls            TINYINT          NULL,
  sequence         TINYINT      NOT NULL,   -- role in the play: sub 1=in 2=out; foul 1=committer 3=fouled; etc.
  position_id      TINYINT          NULL,   -- only on Starting Lineup rows
  position_abbr    VARCHAR(2)       NULL,
  points_scored    TINYINT          NULL,
  play_event_id    TINYINT      NOT NULL,
  play_event       VARCHAR(40)      NULL,
  play_detail_id   SMALLINT         NULL,
  play_detail      VARCHAR(60)      NULL,
  is_blocked       TINYINT(1)   NOT NULL,
  distance         TINYINT      NOT NULL,
  is_fast_break    TINYINT(1)       NULL,
  is_in_the_paint  TINYINT(1)       NULL,
  is_off_turnover  TINYINT(1)       NULL,
  is_second_chance TINYINT(1)       NULL,
  away_score       SMALLINT     NOT NULL,
  home_score       SMALLINT     NOT NULL,
  away_fouls       TINYINT      NOT NULL,
  home_fouls       TINYINT      NOT NULL,
  play_text        VARCHAR(255) NOT NULL,
  PRIMARY KEY (event_id, play_id, play_sequence, player_id),
  KEY ix_pbpp_player (event_id, player_id, play_id),
  KEY ix_pbpp_event  (event_id, play_event)
);
"""

In [122]:
DDL_ROSTERS = """ 
CREATE TABLE IF NOT EXISTS rosters (
  season           SMALLINT     NOT NULL,
  date             DATE         NOT NULL,
  event_id         INT          NOT NULL,
  team_id          INT          NOT NULL,
  team_abbr        VARCHAR(5)   NOT NULL,
  opp_id           INT          NOT NULL,
  opp_abbr         VARCHAR(5)   NOT NULL,
  home             TINYINT(1)   NOT NULL,
  primary_pos_id   TINYINT      NOT NULL,
  primary_pos_abbr VARCHAR(2)   NOT NULL,
  player_id        INT          NOT NULL,
  name             VARCHAR(60)  NOT NULL,
  PRIMARY KEY (event_id, player_id),
  KEY ix_ros_team (event_id, team_id)
);
"""

In [123]:
def get_conn():
    """MySQL connection from .env — nothing hard-coded."""

    conn = mysql.connector.connect(
        host=os.environ["MYSQL_HOST"],
        port=int(os.environ["MYSQL_PORT"]),
        user=os.environ["MYSQL_USER"],
        password=os.environ["MYSQL_PASSWORD"],
        database=os.environ["MYSQL_DATABASE"],
    )
    return conn


def write_table(df: pd.DataFrame, table: str, ddl: str, conn, key: str = "event_id") -> int:
    """Create `table` if needed, then replace each game's rows. Returns rows written."""

    cols = ", ".join(df.columns)
    marks = ", ".join(["%s"] * len(df.columns))
    insert_sql = f"INSERT INTO {table} ({cols}) VALUES ({marks})"

    clean = df.astype(object).where(df.notna(), None)  # NaN -> NULL, numpy scalars -> Python
    with conn.cursor() as cur:
        cur.execute(ddl)
        for eid, game in clean.groupby(key):
            cur.execute(f"DELETE FROM {table} WHERE {key} = %s", (int(eid),))
            cur.executemany(insert_sql, game.to_numpy().tolist())
    conn.commit()
    return len(df)


In [124]:
with get_conn() as conn:
    print("pbp_players_on_court", write_table(on_court[OUT_COLS], "pbp_players_on_court", DDL, conn))
    print("pbp                 ", write_table(pbp, "pbp", DDL_PBP, conn))
    print("pbp_players         ", write_table(pbpp, "pbp_players", DDL_PBP_PLAYERS, conn))
    print("rosters             ", write_table(ros, "rosters", DDL_ROSTERS, conn))

pbp_players_on_court 9930
pbp                  1011
pbp_players          1284
rosters              66


In [106]:
conn = get_conn()
n = write_mysql(on_court, conn)
conn.close()
n

9930

In [125]:
def query(sql: str) -> pd.DataFrame:
    with get_conn() as conn:
        result = pd.read_sql(sql, conn)
    return result

In [129]:
query("""
    SELECT 'on_court' t,
        COUNT(*) n 
    FROM pbp_players_on_court 
    
    UNION ALL 
        
    SELECT 'pbp', 
        COUNT(*) 
    FROM pbp
""")

C:\Users\ChrisBoone\AppData\Local\Temp\ipykernel_47068\2434811033.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(sql, conn)


,t,n
0,on_court,9930
1,pbp,1011


In [137]:
val1_sql = open("sql/validation1.sql").read()
query(val1_sql)


C:\Users\ChrisBoone\AppData\Local\Temp\ipykernel_47068\2434811033.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(sql, conn)


,event_id,team_id,player_id,name,pts_for,pts_against,plays
0,1947312,10,229738,Trevor Ariza,111.0,89.0,362
1,1947312,10,395388,James Harden,108.0,93.0,354
2,1947312,10,329715,Ryan Anderson,95.0,73.0,292
3,1947312,10,229602,P.J. Tucker,92.0,80.0,361
4,1947312,21,845564,Devin Booker,84.0,107.0,306
5,1947312,10,401085,Eric Gordon,81.0,59.0,291
6,1947160,2,280587,Al Horford,77.0,62.0,367
7,1947312,21,467468,Greg Monroe,73.0,75.0,262
8,1947160,9,468895,Draymond Green,73.0,68.0,399
9,1947312,21,660085,T.J. Warren,72.0,96.0,261


In [138]:
val2_sql = open("sql/validation2.sql").read()
query(val2_sql)


C:\Users\ChrisBoone\AppData\Local\Temp\ipykernel_47068\2434811033.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(sql, conn)


,event_id,team_id,team_pts,expected,on_court_pts,diff
0,1947160,2,92.0,460.0,460.0,0.0
1,1947160,9,88.0,440.0,440.0,0.0
2,1947312,10,142.0,710.0,710.0,0.0
3,1947312,21,116.0,580.0,580.0,0.0


In [139]:
query("SELECT COUNT(*) AS n FROM pbp_players_on_court")

C:\Users\ChrisBoone\AppData\Local\Temp\ipykernel_47068\2434811033.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(sql, conn)


,n
0,9930
